## CASE — ¿se puede corregir la estimación de RR en personas sanas y despiertas?
**Problema:** con la FFT congelada, 2 de cada 3 ventanas de CASE dan 6–7 rpm (energía vasomotora lenta).

**Protocolo (fijado antes de ver resultados):**
1. CASE se divide por participantes en **desarrollo (15)** y **prueba (15)**, con una semilla fija.
2. Se evalúan 12 variantes definidas de antemano: modulación {RIIV, RIAV, RIFV, mediana de las 3}
   × límite inferior de la banda {6, 8, 10} rpm (8–10 rpm excluyen la banda vasomotora ~0.1 Hz).
3. Se elige la variante con **menor MAE mediano por participante en desarrollo**, entre las que estiman
   ≥ 90 % de las ventanas válidas (para que no gane una variante por abstenerse).
4. Esa variante se aplica sin cambios a CASE prueba, **BIDMC** y **CapnoBase**, para comprobar que
   no empeora donde el método original ya funcionaba.

In [ ]:
# x-1 — Imports, configuración y rutas
import json
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MAX = CFG["rpm_max"]
N = VENTANA_S * FS
LIMITES_INF = [6, 8, 10]
MODULACIONES = ["RIIV", "RIAV", "RIFV", "Mediana 3"]

CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated" / "physiological"
CAPNO_DIR = BASE_DIR / "data" / "capnobase" / "csv"
CASE = sorted(CASE_DIR.glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1]))
BIDMC = [f"bidmc{i:02d}" for i in range(1, 54)]
CAPNO = sorted(p.name.replace("_signal.csv", "") for p in CAPNO_DIR.glob("*_signal.csv"))

rng = np.random.default_rng(2026)
perm = [str(x) for x in rng.permutation([p.stem for p in CASE])]
CASE_DEV = set(perm[:len(perm) // 2])
print(f"CASE: {len(CASE)} ({len(CASE_DEV)} desarrollo) | BIDMC: {len(BIDMC)} | CapnoBase: {len(CAPNO)}")
print("Desarrollo:", sorted(CASE_DEV, key=lambda s: int(s.split('_')[1])))

In [ ]:
# x-2 — Estimadores (todas las variantes por ventana)
def pico(x, fs, nfft, lo):
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, 1 / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    b = (f >= lo) & (f <= RPM_MAX)
    return f[np.argmax(np.where(b, P, -np.inf))]

def series_latidos(ppg):
    b, a = butter(2, [0.5, 8], btype="band", fs=FS)
    x = filtfilt(b, a, ppg)
    esc = 1.4826 * np.median(np.abs(x - np.median(x)))
    pk, _ = find_peaks(x, distance=int(0.33 * FS), prominence=0.5 * esc)
    vl = np.array([p0 + np.argmin(x[p0:p1]) for p0, p1 in zip(pk[:-1], pk[1:])], int)
    pk = pk[1:]
    t = pk / FS
    ibi = np.diff(t)
    t, pk, vl = t[1:], pk[1:], vl[1:]
    med = pd.Series(ibi).rolling(11, center=True, min_periods=3).median().to_numpy()
    ok = (ibi >= 0.33) & (ibi <= 1.5) & (np.abs(ibi - med) <= 0.3 * med)
    y = ppg - np.median(ppg)
    return t[ok], (y[pk] - y[vl])[ok], ibi[ok]

def rr_serie(t, v, t0, lo):
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < 15:
        return np.nan
    tg = t0 + np.arange(VENTANA_S * 4) / 4
    return pico(np.interp(tg, t[m], v[m]), 4, 1024, lo)

def procesar(ppg):
    """Devuelve una función t0 -> dict con las 12 variantes y la excursión (artefactos)."""
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv_art = filtfilt(b, a, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv_art[i:i + n5]) for i in range(0, len(ppg) - n5 + 1, n5)])
    t, iav, ifv = series_latidos(ppg)
    q = FS // 5

    def ventana(t0):
        i0 = int(round(t0 * FS))
        x5 = decimate(decimate(riiv[i0:i0 + N], 5, zero_phase=True), q // 5, zero_phase=True)
        out = {"Excursión": max(np.std(riiv_art[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico}
        for lo in LIMITES_INF:
            e = {"RIIV": pico(x5, FS / q, 2048, lo), "RIAV": rr_serie(t, iav, t0, lo),
                 "RIFV": rr_serie(t, ifv, t0, lo)}
            e["Mediana 3"] = np.nanmedian(list(e.values())) if np.isfinite(list(e.values())).sum() >= 2 else np.nan
            out.update({f"{m} ≥{lo}": v for m, v in e.items()})
        return out
    return ventana

def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

In [ ]:
# x-3 — CASE
CATEGORIA = {1: "divertido", 2: "divertido", 3: "aburrido", 4: "aburrido",
             5: "relajante", 6: "relajante", 7: "miedo", 8: "miedo"}

def marcas_banda(rsp, fs=1000):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=int(f), prominence=0.25 * iqr)
    return pk / f

filas = []
for p in CASE:
    d = pd.read_csv(p, usecols=["bvp", "rsp", "video"])
    vid = d["video"].to_numpy(int)
    ppg = resample_poly(d["bvp"].to_numpy(float), FS, 1000)
    rsp125 = resample_poly(d["rsp"].to_numpy(float), FS, 1000)
    marcas = marcas_banda(d["rsp"].to_numpy(float))
    ventana = procesar(ppg)
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        v = vid[int(t0 * 1000):int((t0 + VENTANA_S) * 1000)]
        if v.min() != v.max() or v[0] not in CATEGORIA:
            continue
        i0 = int(round(t0 * FS))
        ref = rr_ref(marcas, t0, t0 + VENTANA_S)
        x5 = decimate(decimate(rsp125[i0:i0 + N], 5, zero_phase=True), 5, zero_phase=True)
        fiable = np.isfinite(ref) and 6 <= ref <= RPM_MAX and abs(ref - pico(x5, 5, 2048, 6)) <= 2
        filas.append({"Base": "CASE", "Sujeto": p.stem,
                      "Conjunto": "desarrollo" if p.stem in CASE_DEV else "prueba",
                      "Inicio (s)": t0, "RR referencia": ref, "Ref fiable": fiable, **ventana(t0)})
D_case = pd.DataFrame(filas)
print(len(D_case), "ventanas CASE")

In [ ]:
# x-4 — BIDMC y CapnoBase (mismo criterio de validez que en los notebooks 02 y 03)
def _lista(c):
    return np.array([]) if pd.isna(c) or str(c).strip() == "" else np.array(str(c).split(), dtype=float)

filas = []
for rid in BIDMC:
    ruta = str(BASE_DIR / "data" / rid)
    rec = wfdb.rdrecord(ruta)
    ppg = rec.p_signal[:, [s.strip(" ,").upper() for s in rec.sig_name].index("PLETH")]
    ann = wfdb.rdann(ruta, "breath")
    aux = np.array([(a or "").strip(" \x00") for a in ann.aux_note])
    m1, m2 = (np.sort(ann.sample[aux == k]) / (ann.fs or rec.fs) for k in ("ann1", "ann2"))
    ventana = procesar(ppg)
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        r1, r2 = rr_ref(m1, t0, t0 + VENTANA_S), rr_ref(m2, t0, t0 + VENTANA_S)
        filas.append({"Base": "BIDMC", "Sujeto": rid, "Conjunto": "BIDMC", "Inicio (s)": t0,
                      "RR referencia": r1, "Ref fiable": bool(abs(r1 - r2) <= CFG["umbral_anot"]),
                      **ventana(t0)})
for c in CAPNO:
    sig = pd.read_csv(CAPNO_DIR / f"{c}_signal.csv")
    par = pd.read_csv(CAPNO_DIR / f"{c}_param.csv").iloc[0]
    lab = pd.read_csv(CAPNO_DIR / f"{c}_labels.csv", keep_default_na=False).iloc[0]
    ppg = resample_poly(sig["pleth_y"].to_numpy(float), FS, int(par["samplingrate_pleth"]))
    m1 = np.sort(_lista(lab["co2_startexp_x"])) / int(par["samplingrate_co2"])
    ventana = procesar(ppg)
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        r1 = rr_ref(m1, t0, t0 + VENTANA_S)
        filas.append({"Base": "CapnoBase", "Sujeto": c, "Conjunto": "CapnoBase", "Inicio (s)": t0,
                      "RR referencia": r1, "Ref fiable": bool(np.isfinite(r1)), **ventana(t0)})
D = pd.concat([D_case, pd.DataFrame(filas)], ignore_index=True)
D["Válida"] = D["Ref fiable"] & (D["Excursión"] <= CFG["umbral_art"]) & D["RR referencia"].notna()
D.to_csv(OUT / "case_correccion_ventanas.csv", index=False)
print(D.groupby("Conjunto")["Válida"].agg(["size", "sum"]))

In [ ]:
# x-5 — Selección en CASE desarrollo
VARIANTES = [f"{m} ≥{lo}" for lo in LIMITES_INF for m in MODULACIONES]

def mae_mediano(g, col):
    e = (g[col] - g["RR referencia"]).abs()
    return e.groupby(g["Sujeto"]).mean().median(), 100 * (e <= 2).mean(), 100 * g[col].notna().mean()

dev = D[(D["Conjunto"] == "desarrollo") & D["Válida"]]
sel = pd.DataFrame([{"Variante": v, **dict(zip(["MAE mediano", "% ≤ 2 rpm", "Cobertura %"],
                                               mae_mediano(dev, v)))} for v in VARIANTES])
print(sel.round(2).sort_values("MAE mediano").to_string(index=False))
cand = sel[sel["Cobertura %"] >= 90]
ELEGIDA = cand.loc[cand["MAE mediano"].idxmin(), "Variante"]
print(f"\nVariante elegida en desarrollo: {ELEGIDA}")
(OUT / "config_case_correccion.json").write_text(json.dumps({"variante": ELEGIDA, "semilla_division": 2026,
                                                             "case_desarrollo": sorted(CASE_DEV)}, indent=2))

In [ ]:
# x-6 — Prueba: CASE prueba, BIDMC y CapnoBase con la variante elegida frente a la original
ORIGINAL = "RIIV ≥6"                      # = FFT congelada de los notebooks 02/03
filas = []
for conj in ["desarrollo", "prueba", "BIDMC", "CapnoBase"]:
    g = D[(D["Conjunto"] == conj) & D["Válida"]]
    for nombre, col in [("Original (RIIV ≥6)", ORIGINAL), (f"Elegida ({ELEGIDA})", ELEGIDA)]:
        mm, p2, cob = mae_mediano(g, col)
        filas.append({"Conjunto": conj, "Método": nombre, "Sujetos": g["Sujeto"].nunique(),
                      "MAE mediano": mm, "% ≤ 2 rpm": p2, "Cobertura %": cob})
R = pd.DataFrame(filas)
R.to_csv(OUT / "case_correccion_resumen.csv", index=False)
print(R.round(2).to_string(index=False))
print("\nNota: 'desarrollo' se usó para elegir; la evaluación honesta es 'prueba', BIDMC y CapnoBase.")